# AlphaTrain Pillar 2Q-v3: Density Reward (Break the Blob)

Fix the value blob: r(t) = 1.0 + empty/81 + score/10, γ=0.98.
IQR 5.0 (was 0.75), blob 64% (was 97%). Value head can now discriminate.

- Pure v5 data (1,953 games, 4.68M states, mean 5,117)
- γ=0.98, max_score=100, 128 bins (0.79/bin)
- val_weight=0.01, rank_weight=2.0 (stronger ranking)
- 50% endgame oversampling (was 30%)
- 15 epochs, warm start from 2P ep6

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v5_density_g098.pt.gz` — density reward data
3. `pillar2p_ep6.pt` — base model (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying model...')
shutil.copy(f'{DRIVE}/pillar2p_ep6.pt', '/content/alphatrain/data/model.pt')
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress density reward data
print('Decompressing density reward data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v5_density_g098.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

# Sanity check distribution
import torch
d = torch.load('/content/alphatrain/data/selfplay.pt', weights_only=True, map_location='cpu')
bins = torch.linspace(0, float(d['max_score']), int(d['num_value_bins']))
vals = (d['val_targets'] * bins).sum(dim=-1)
iqr = torch.quantile(vals.float(), 0.75) - torch.quantile(vals.float(), 0.25)
print(f'\nValue IQR: {iqr:.1f} (must be >2.0, was 0.75 before fix)')
assert iqr > 2.0, f'Value blob NOT fixed! IQR={iqr:.2f}'
print(f'Blob (V>85): {100*(vals>85).float().mean():.0f}% (must be <80%, was 97%)')
del d, vals

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2Q-v3: Density reward, γ=0.98, rank_weight=2.0, 50% endgame
# Warm start from 2P epoch 6
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 2.0 \
    --endgame-fraction 0.5 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 15 --batch-size 12288 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2q_best.pt \
    --save-dir /content/checkpoints/pillar2q

In [ ]:
# Copy ALL epoch checkpoints to Drive for bracket eval
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2q/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2q_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2q/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2q_{f}')